# Bitcoin Feature Selection

Cleaned notebook for feature-importance experiments used before the GRU modeling stage. Colab metadata, local paths, and execution outputs were removed for publication.

# import

In [ ]:
# 파일시스템, 시간 패키지
import os
import time, datetime

#시간 계산
def clock(start):
    sec = time.time() - start #현재시간 - 시스템초기시간
    times = str(datetime.timedelta(seconds = sec)).split(".") # 시간:분:초로 변환
    times = times[0]
    return times

# 넘파이, 판다스
import scipy as sp
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 500) # 열 전체 다 나오도록 해주는 함수

# 시각화 : 맷플롯, 씨본
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# jupyter에 matplotlib을 바로 띄워주는 매직함수
%matplotlib inline 

#경고 무시
import warnings 
warnings.filterwarnings(action = 'ignore') 

from sklearn.preprocessing import StandardScaler
from sklearn import model_selection as ms

# Time 변수를 시간형식으로 변경하는 함수
def shift_time_letter(str_list) :
  result = []
  for i in range(len(str_list)):
    if str_list[i][-2:]=='AM' or str_list[i][-2:]=='PM': # AM/PM이 마지막에 있는 경우
      result = str_list
    elif str_list[i][11:13]=='AM' or str_list[i][11:13]=='PM': # AM/PM이 가운데 있는 경우
      d_merge = str_list[i][:10] + ' ' + str_list[i][14:] + ' ' +str_list[i][11:13]
      result.append(d_merge)
    else: # 그 외는 패스
      pass
  return result

In [ ]:
df=pd.read_csv("../data/final_data_0601_3.csv",encoding='cp949')
df

# 모델링

In [ ]:
col_list=df.iloc[:,2:].columns
col_list

In [ ]:
X = df.iloc[:,2:]

col_list=X.columns
col_list

In [ ]:
X.info()

In [ ]:
# 전처리

y_target= df.iloc[:,1]

# 데이터 정규화 (MinMax정규화)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scale= scaler.fit_transform(X)
X=pd.DataFrame(X_scale,columns=col_list)

# train/test
X_train, X_test, y_train, y_test = ms.train_test_split(X, y_target, 
                                                      test_size = 0.2, random_state = 100, stratify = y_target )

In [ ]:
X_train.head(1)

### 1)RF

In [ ]:
start = time.time()

# 랜덤으로 할 파라미터 정의
param_list = {"n_estimators": list(range(20, 200, 20)),
              "max_depth": list(range(4, 21, 4)),
              "max_features": list(range(5, 40, 5)),
              "min_samples_split": list(range(3, 13, 2))}

# 하이퍼파라미터 최적화
RF = RandomForestClassifier()
RF_random_search = RandomizedSearchCV(estimator = RF,
                                        param_distributions = param_list,
                                        n_iter = 3,       # 5번 반복하는 랜덤포레스트를 구현
                                        cv = 3,           # 3번의 cross-validation
                                        n_jobs = 10,
                                        random_state=42) 
RF_random_search.fit(X_train, y_train)
y_pred = RF_random_search.predict(X_test)

#성능평가
print('accuracy',mt.accuracy_score(y_test,y_pred))

print( clock(start) )
print( RF_random_search.best_params_ ) #파라미터 중 가장 정확도가 높은 파라미터를 출력
#가장 추정이 잘된 변수명들의 정확도를 순서대로 나열함
f_i1 = pd.DataFrame(sorted(zip(RF_random_search.best_estimator_.feature_importances_*100, X_train.columns), reverse=True), columns=['f_i','columns'])
f_i1.head(20)

In [ ]:
f_i1.head(20).loc[:,'columns']

### 2)XGB

In [ ]:
start = time.time()

XGB = xgb.XGBClassifier()

param_list = {"n_estimators": list(range(10, 300, 10)),
              "max_depth": list(range(4, 21, 4)),
              "max_features": list(range(3, 53, 5)),
              "min_samples_split": list(range(3, 13, 2))}

# 하이퍼파라미터 최적화
XGB_random_search = RandomizedSearchCV(estimator = XGB,
                                        param_distributions = param_list,
                                        n_iter = 5,       # 5번반복하는 xgboost를 구현
                                        cv = 3,           # cross-validation 3번 반복
                                        n_jobs = 10,          
                                        random_state=1)
XGB_random_search.fit(X_train, y_train)
y_pred = XGB_random_search.predict(X_test) 

#성능평가
print('accuracy',mt.accuracy_score(y_test,y_pred) )  #multi calss 일때 사용, 기본="binary",  none : class 각각 실행 / micro : 클래스 합쳐서 실행
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print( XGB_random_search.best_params_ ) 
print( clock(start) )

# 가장 정확도가 높은 변수명 순으로 나열
f_i2 = pd.DataFrame(sorted(zip(XGB_random_search.best_estimator_.feature_importances_*100, X_train.columns), reverse=True), columns=['f_i','columns'])
f_i2.head(20)

### 3)GBM

In [ ]:
start = time.time()


GB = GradientBoostingClassifier(learning_rate=0.05)

# 하이퍼파라미터 최적화
GB_random_search = RandomizedSearchCV(estimator = GB,
                                        param_distributions = param_list,
                                        n_iter = 5,       # 5번반복하는 그라디언트 부스팅을 구현
                                        cv = 3,           # cross-validation 3번 반복
                                        n_jobs = 10,
                                        random_state=42)

GB_random_search.fit(X_train, y_train)
y_pred = GB_random_search.predict(X_test)

#성능평가
#성능평가
print('accuracy',mt.accuracy_score(y_test,y_pred) )  #multi calss 일때 사용, 기본="binary",  none : class 각각 실행 / micro : 클래스 합쳐서 실행
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print( GB_random_search.best_params_ ) 
print( clock(start) )

# 가장 정확도가 높은 변수명 순으로 나열
f_i3 = pd.DataFrame(sorted(zip(GB_random_search.best_estimator_.feature_importances_*100, X_train.columns), reverse=True), columns=['f_i','columns'])
f_i3.head(20)

### 4)LGBM

In [ ]:
start = time.time()

# 부스팅 타입은 default값인 gbdt로, 학습률은 0.05로 지정 하였음 =>128
# 부스팅 타입은 default값인 gbdt로, 학습률은 0.01로 지정 하였음 =>194
# 부스팅 타입은 default값인 gbdt로, 학습률은 0.01로 지정 하였음 =>105.00471775373902
LGB = lgb.LGBMClassifier(learning_rate = 0.1)
param_list = {"n_estimators": list(range(10, 300, 10)),
              "max_depth": list(range(4, 21, 4)),
              "max_features": list(range(3, 50, 2)),
              "min_samples_split": list(range(3, 13, 2))}

# 하이퍼파라미터 최적화
LGB_random_search = RandomizedSearchCV(estimator = LGB,
                                        param_distributions = param_list,
                                        n_iter = 10,      # 10번반복하는 lightgbm 구현 : 성능개선 시도
                                        cv = 3,           # cross-validation 3번 반복
                                        n_jobs = 10,
                                        random_state=42)

LGB_random_search.fit(X_train, y_train)
lgbm_y_pred1 = LGB_random_search.predict(X_test)

#성능평가
print('accuracy',mt.accuracy_score(y_test,y_pred) )  #multi calss 일때 사용, 기본="binary",  none : class 각각 실행 / micro : 클래스 합쳐서 실행
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

#print( LGB_random_search.best_params_ )
print( clock(start) )

# 가장 정확도가 높은 변수명 순으로 나열
f_i4 = pd.DataFrame(sorted(zip(LGB_random_search.best_estimator_.feature_importances_*100, X_train.columns), reverse=True), columns=['f_i','columns'])
f_i4.head(20)

In [ ]:
top15_list=['%K','%B','mpi','outflow_mean','market_premium','%D','RS','volume_y','volume','outflow_total','nvt_golden_cross','value','stock_to_flow',' volume _x ','AU']


In [ ]:
df

In [ ]:
pd.DataFrame(pd.concat((f_i1['columns'].head(15), f_i2['columns'].head(1), f_i3['columns'].head(11), f_i4['columns'].head(8)), axis=0).value_counts())

In [ ]:
pd.concat((f_i1.head(20), f_i2.head(1), f_i3.head(11), f_i4.head(8)), axis=0)